# 🔧 數據清理工具包集成系統 / Data Cleaning Tools Integration System

## 📋 系統概述 / System Overview

本Notebook提供完整的數據清理和分析工具包，專為Simplified System量化交易平台設計。
This notebook provides a comprehensive data cleaning and analysis toolkit specifically designed for the Simplified System quantitative trading platform.

### 🎯 核心功能 / Core Features
- **自動化數據質量分析** - 使用 pandas-profiling
- **缺失值可視化** - 使用 missingno
- **智能數據比較** - 使用 sweetviz
- **0700.HK專用分析** - 集成真實港股數據
- **中英文雙語支持** - 專業級報告生成

### 🚀 快速開始 / Quick Start
1. 運行所有單元格安裝依賴
2. 加載0700.HK數據進行分析
3. 生成專業數據質量報告

In [ ]:
# Environment Setup / 環境設置
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Add Simplified System path / 添加Simplified System路徑
sys.path.append('.')
sys.path.append('src')

# Basic imports / 基礎導入
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from pathlib import Path

# Data cleaning tools / 數據清理工具
try:
    import ydata_profiling
    from ydata_profiling import ProfileReport
    print("✅ ydata-profiling imported successfully")
except ImportError:
    print("❌ ydata-profiling not installed. Please run: pip install ydata-profiling")

try:
    import missingno as msno
    print("✅ missingno imported successfully")
except ImportError:
    print("❌ missingno not installed. Please run: pip install missingno")

try:
    import sweetviz as sv
    print("✅ sweetviz imported successfully")
except ImportError:
    print("❌ sweetviz not installed. Please run: pip install sweetviz")

# Set plot style / 設置圖表樣式
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set display options / 設置顯示選項
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

print("🔧 Environment setup completed! / 環境設置完成！")
print(f"📊 Pandas version: {pd.__version__}")
print(f"🐍 Python version: {sys.version.split()[0]}")

## 📡 數據加載模塊 / Data Loading Module

集成Simplified System的真實數據源
Integration with Simplified System real data sources

In [ ]:
class DataLoadingManager:
    """數據加載管理器 / Data Loading Manager"""
    
    def __init__(self):
        self.data_dir = Path('data')
        self.stock_data = None
        self.government_data = None
        
    def load_0700_hk_data(self, source='api'):
        """加載0700.HK數據 / Load 0700.HK data"""
        try:
            # Method 1: Load from Simplified System API
            if source == 'api':
                try:
                    from src.api.stock_api import get_stock_prices_dataframe
                    print("📡 Loading data from Simplified System API...")
                    data = get_stock_prices_dataframe('0700.HK', 1095)
                    if data is not None:
                        self.stock_data = self._standardize_stock_data(data)
                        return self.stock_data
                except Exception as e:
                    print(f"⚠️ API loading failed: {e}, trying local files...")
            
            # Method 2: Load from local CSV files
            local_sources = [
                '0700_results_20251125_181239.csv',
                '0700_results_20251125_181639.csv',
                'data/0700_hk.csv',
                'data/stocks/0700.HK.csv'
            ]
            
            for source_file in local_sources:
                if Path(source_file).exists():
                    print(f"📁 Loading from local file: {source_file}")
                    data = pd.read_csv(source_file, index_col=0, parse_dates=True)
                    self.stock_data = self._standardize_stock_data(data)
                    return self.stock_data
            
            # Method 3: Generate sample data if no real data found
            print("⚠️ No real data found, generating sample 0700.HK data...")
            self.stock_data = self._generate_sample_0700_data()
            return self.stock_data
            
        except Exception as e:
            print(f"❌ Error loading data: {e}")
            return None
    
    def _standardize_stock_data(self, data):
        """標準化股票數據格式 / Standardize stock data format"""
        df = data.copy()
        
        # Ensure datetime index
        if not isinstance(df.index, pd.DatetimeIndex):
            if 'date' in df.columns:
                df['date'] = pd.to_datetime(df['date'])
                df.set_index('date', inplace=True)
            else:
                df.index = pd.to_datetime(df.index)
        
        # Standardize column names
        column_mapping = {
            'Price': 'Close', 'price': 'Close',
            'Volume': 'Volume', 'volume': 'Volume',
            'Open': 'Open', 'open': 'Open',
            'High': 'High', 'high': 'High',
            'Low': 'Low', 'low': 'Low'
        }
        
        df = df.rename(columns=column_mapping)
        
        # Ensure required columns exist
        if 'Close' not in df.columns:
            if len(df.columns) > 0:
                df['Close'] = df.iloc[:, 0]
            else:
                raise ValueError("No price data found")
        
        # Add missing OHLC columns if needed
        for col in ['Open', 'High', 'Low']:
            if col not in df.columns:
                df[col] = df['Close'] * (1 + np.random.normal(0, 0.01, len(df)))
        
        if 'Volume' not in df.columns:
            df['Volume'] = np.random.randint(10000000, 30000000, len(df))
        
        # Sort by date
        df = df.sort_index()
        
        print(f"✅ Data standardized: {len(df)} records from {df.index[0].date()} to {df.index[-1].date()}")
        return df
    
    def _generate_sample_0700_data(self):
        """生成示例0700.HK數據 / Generate sample 0700.HK data"""
        np.random.seed(42)
        dates = pd.date_range('2022-01-01', '2025-11-24', freq='D')
        
        # 基於騰訊真實價格趨勢 / Based on Tencent real price trends
        base_price = 400
        trend = np.linspace(base_price, base_price * 1.7, len(dates))
        volatility = np.random.randn(len(dates)) * 12
        price = trend + volatility
        price = np.maximum(price, 50)
        
        data = pd.DataFrame({
            'Open': price * (1 + np.random.randn(len(dates)) * 0.015),
            'High': price * (1 + np.random.rand(len(dates)) * 0.025),
            'Low': price * (1 - np.random.rand(len(dates)) * 0.025),
            'Close': price,
            'Volume': np.random.randint(15000000, 35000000, len(dates))
        }, index=dates)
        
        # 添加一些真實的價格模式 / Add some realistic price patterns
        data['Return'] = data['Close'].pct_change()
        data['MA20'] = data['Close'].rolling(20).mean()
        data['Volatility'] = data['Return'].rolling(20).std()
        
        return data
    
    def load_government_data(self, data_type='hibor'):
        """加載政府經濟數據 / Load government economic data"""
        try:
            gov_data_dir = self.data_dir / 'government'
            if not gov_data_dir.exists():
                gov_data_dir.mkdir(parents=True, exist_ok=True)
            
            # Look for government data files
            gov_files = list(gov_data_dir.glob(f'*{data_type}*.json'))
            
            if gov_files:
                latest_file = max(gov_files, key=lambda x: x.stat().st_mtime)
                print(f"📊 Loading government data from: {latest_file}")
                
                import json
                with open(latest_file, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                
                if 'data' in data:
                    df = pd.DataFrame(data['data'])
                    self.government_data = df
                    return df
            
            print(f"⚠️ No {data_type} data found")
            return None
            
        except Exception as e:
            print(f"❌ Error loading government data: {e}")
            return None

# Initialize data loader / 初始化數據加載器
data_loader = DataLoadingManager()
print("🚀 Data loading manager initialized!")

In [ ]:
# Load 0700.HK data / 加載0700.HK數據
print("📈 Loading 0700.HK data...")
stock_data = data_loader.load_0700_hk_data()

if stock_data is not None:
    print(f"✅ Stock data loaded successfully!")
    print(f"📊 Records: {len(stock_data)}")
    print(f"📅 Date range: {stock_data.index[0].date()} to {stock_data.index[-1].date()}")
    print(f"💰 Price range: {stock_data['Close'].min():.2f} - {stock_data['Close'].max():.2f} HKD")
    print(f"📈 Average volume: {stock_data['Volume'].mean():,.0f}")
    
    # Display basic info / 顯示基本信息
    print("\n📋 Data Info:")
    stock_data.info()
    
    print("\n📊 Sample Data (first 5 rows):")
    display(stock_data.head())
else:
    print("❌ Failed to load stock data")

## 🔍 數據質量分析工具包 / Data Quality Analysis Toolkit

In [ ]:
class DataQualityAnalyzer:
    """數據質量分析器 / Data Quality Analyzer"""
    
    def __init__(self, data):
        self.data = data
        self.quality_report = {}
    
    def basic_quality_check(self):
        """基礎質量檢查 / Basic Quality Check"""
        print("🔍 Basic Data Quality Check / 基礎數據質量檢查")
        print("=" * 60)
        
        # Data shape / 數據形狀
        print(f"📊 Data shape: {self.data.shape}")
        
        # Missing values / 缺失值
        missing_values = self.data.isnull().sum()
        total_missing = missing_values.sum()
        print(f"❓ Missing values: {total_missing} ({total_missing/self.data.size*100:.2f}%)")
        
        if total_missing > 0:
            print("Missing by column:")
            for col, missing in missing_values[missing_values > 0].items():
                print(f"  {col}: {missing} ({missing/len(self.data)*100:.2f}%)")
        
        # Duplicate rows / 重複行
        duplicates = self.data.duplicated().sum()
        print(f"🔄 Duplicate rows: {duplicates}")
        
        # Data types / 數據類型
        print("\n📝 Data types:")
        print(self.data.dtypes)
        
        # Memory usage / 內存使用
        memory_usage = self.data.memory_usage(deep=True).sum() / 1024 / 1024
        print(f"💾 Memory usage: {memory_usage:.2f} MB")
        
        # Store results / 存儲結果
        self.quality_report['basic'] = {
            'shape': self.data.shape,
            'missing_values': total_missing,
            'missing_percentage': total_missing/self.data.size*100,
            'duplicates': duplicates,
            'memory_usage_mb': memory_usage
        }
        
        return self.quality_report['basic']
    
    def statistical_quality_check(self):
        """統計質量檢查 / Statistical Quality Check"""
        print("\n📈 Statistical Quality Check / 統計質量檢查")
        print("=" * 60)
        
        numeric_cols = self.data.select_dtypes(include=[np.number]).columns
        
        for col in numeric_cols:
            print(f"\n📊 {col}:")
            
            # Basic statistics / 基礎統計
            values = self.data[col].dropna()
            print(f"  Mean: {values.mean():.4f}")
            print(f"  Std: {values.std():.4f}")
            print(f"  Min: {values.min():.4f}")
            print(f"  Max: {values.max():.4f}")
            print(f"  Median: {values.median():.4f}")
            
            # Outliers / 異常值
            Q1 = values.quantile(0.25)
            Q3 = values.quantile(0.75)
            IQR = Q3 - Q1
            outliers = ((values < (Q1 - 1.5 * IQR)) | (values > (Q3 + 1.5 * IQR))).sum()
            print(f"  Outliers: {outliers} ({outliers/len(values)*100:.2f}%)")
            
            # Skewness / 偏度
            skewness = values.skew()
            print(f"  Skewness: {skewness:.4f} ({'Right-skewed' if skewness > 0 else 'Left-skewed' if skewness < 0 else 'Symmetric'})")
    
    def time_series_quality_check(self):
        """時間序列質量檢查 / Time Series Quality Check"""
        if not isinstance(self.data.index, pd.DatetimeIndex):
            print("⚠️ Data is not time series indexed")
            return
        
        print("\n⏰ Time Series Quality Check / 時間序列質量檢查")
        print("=" * 60)
        
        # Date range / 日期範圍
        date_range = self.data.index.max() - self.data.index.min()
        print(f"📅 Date range: {date_range.days} days")
        
        # Frequency / 頻率
        inferred_freq = pd.infer_freq(self.data.index)
        print(f"🔄 Inferred frequency: {inferred_freq or 'Irregular'}")
        
        # Gaps in data / 數據間隙
        expected_dates = pd.date_range(self.data.index.min(), self.data.index.max(), freq='D')
        missing_dates = expected_dates.difference(self.data.index)
        print(f"❓ Missing dates: {len(missing_dates)}")
        
        if len(missing_dates) > 0:
            print(f"  First missing date: {missing_dates[0].date()}")
            print(f"  Last missing date: {missing_dates[-1].date()}")
            
        # Weekend/holiday patterns / 周末/假期模式
        weekdays = self.data.index.dayofweek
        print(f"📊 Weekday distribution: {pd.Series(weekdays).value_counts().sort_index().to_dict()}")

# Perform data quality analysis / 執行數據質量分析
if stock_data is not None:
    quality_analyzer = DataQualityAnalyzer(stock_data)
    
    # Basic quality check / 基礎質量檢查
    basic_quality = quality_analyzer.basic_quality_check()
    
    # Statistical quality check / 統計質量檢查
    quality_analyzer.statistical_quality_check()
    
    # Time series quality check / 時間序列質量檢查
    quality_analyzer.time_series_quality_check()
else:
    print("❌ No data available for quality analysis")

## 📊 缺失值可視化分析 / Missing Data Visualization Analysis

In [ ]:
def analyze_missing_data(data):
    """缺失值分析 / Missing Data Analysis"""
    print("🔍 Missing Data Analysis / 缺失值分析")
    print("=" * 50)
    
    # Create missing data summary / 創建缺失值摘要
    missing_summary = data.isnull().sum()
    missing_percentage = (missing_summary / len(data)) * 100
    
    missing_df = pd.DataFrame({
        'Missing Count': missing_summary,
        'Missing Percentage': missing_percentage
    }).sort_values('Missing Count', ascending=False)
    
    if missing_df['Missing Count'].sum() > 0:
        print("Missing Data Summary:")
        display(missing_df[missing_df['Missing Count'] > 0])
        
        # Missing data visualization / 缺失值可視化
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        fig.suptitle('Missing Data Analysis / 缺失值分析', fontsize=16)
        
        # 1. Matrix plot / 矩陣圖
        try:
            import missingno as msno
            msno.matrix(data, ax=axes[0, 0], sparkline=False)
            axes[0, 0].set_title('Missing Data Matrix / 缺失值矩陣')
        except ImportError:
            sns.heatmap(data.isnull(), ax=axes[0, 0], cbar=True, yticklabels=False)
            axes[0, 0].set_title('Missing Data Heatmap / 缺失值熱力圖')
        
        # 2. Bar plot / 條形圖
        missing_df[missing_df['Missing Count'] > 0]['Missing Count'].plot(
            kind='bar', ax=axes[0, 1]
        )
        axes[0, 1].set_title('Missing Count by Column / 每列缺失值數量')
        axes[0, 1].set_xlabel('Columns')
        axes[0, 1].set_ylabel('Missing Count')
        axes[0, 1].tick_params(axis='x', rotation=45)
        
        # 3. Missing percentage / 缺失值百分比
        missing_df[missing_df['Missing Count'] > 0]['Missing Percentage'].plot(
            kind='bar', ax=axes[1, 0], color='coral'
        )
        axes[1, 0].set_title('Missing Percentage by Column / 每列缺失值百分比')
        axes[1, 0].set_xlabel('Columns')
        axes[1, 0].set_ylabel('Missing Percentage (%)')
        axes[1, 0].tick_params(axis='x', rotation=45)
        
        # 4. Data completeness / 數據完整性
        completeness = 100 - missing_percentage
        completeness[completeness.index.isin(data.columns)].plot(
            kind='bar', ax=axes[1, 1], color='lightgreen'
        )
        axes[1, 1].set_title('Data Completeness / 數據完整性')
        axes[1, 1].set_xlabel('Columns')
        axes[1, 1].set_ylabel('Completeness (%)')
        axes[1, 1].tick_params(axis='x', rotation=45)
        axes[1, 1].axhline(y=95, color='red', linestyle='--', alpha=0.7, label='95% Threshold')
        axes[1, 1].legend()
        
        plt.tight_layout()
        plt.show()
        
    else:
        print("✅ No missing data found! / 未發現缺失值！")
        
        # Show data completeness plot / 顯示數據完整性圖
        plt.figure(figsize=(10, 6))
        completeness = pd.Series([100] * len(data.columns), index=data.columns)
        completeness.plot(kind='bar', color='lightgreen')
        plt.title('Data Completeness / 數據完整性')
        plt.xlabel('Columns')
        plt.ylabel('Completeness (%)')
        plt.xticks(rotation=45)
        plt.ylim(0, 100)
        plt.show()

# Perform missing data analysis / 執行缺失值分析
if stock_data is not None:
    analyze_missing_data(stock_data)
else:
    print("❌ No data available for missing data analysis")

## 📈 專業數據分析報告生成 / Professional Data Analysis Report Generation

In [ ]:
def generate_profiling_report(data, title="0700.HK Data Analysis"):
    """生成專業數據分析報告 / Generate Professional Data Analysis Report"""
    try:
        from ydata_profiling import ProfileReport
        
        print(f"📊 Generating {title} profiling report...")
        
        # Create profile report / 創建分析報告
        profile = ProfileReport(
            data,
            title=f"{title} / 數據分析報告",
            explorative=True,
            dark_mode=False,
            minimal=False
        )
        
        # Save report / 保存報告
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        report_file = f"data_analysis_report_{timestamp}.html"
        
        profile.to_file(report_file)
        print(f"✅ Report saved as: {report_file}")
        
        # Display report in notebook / 在Notebook中顯示報告
        display(profile)
        
        return profile, report_file
        
    except ImportError:
        print("❌ ydata-profiling not available. Please install with: pip install ydata-profiling")
        return None, None
    except Exception as e:
        print(f"❌ Error generating report: {e}")
        return None, None

# Generate profiling report / 生成分析報告
if stock_data is not None:
    profile_report, report_file = generate_profiling_report(
        stock_data, 
        "0700.HK Trading Data Analysis / 0700.HK 交易數據分析"
    )
else:
    print("❌ No data available for profiling report")

## 🔄 智能數據比較分析 / Smart Data Comparison Analysis

In [ ]:
def generate_comparison_report(data1, data2=None, data1_name="Original", data2_name="Processed"):
    """生成智能數據比較報告 / Generate Smart Data Comparison Report"""
    try:
        import sweetviz as sv
        
        print(f"🔄 Generating comparison report: {data1_name} vs {data2_name}")
        
        if data2 is None:
            # Generate single dataset analysis / 生成單數據集分析
            report = sv.analyze(
                data1,
                target_feat=None,
                pairwise_analysis="auto"
            )
            report_title = f"{data1_name} Data Analysis Report"
        else:
            # Generate comparison report / 生成比較報告
            report = sv.compare(
                [data1, data1_name],
                [data2, data2_name],
                target_feat=None
            )
            report_title = f"{data1_name} vs {data2_name} Comparison Report"
        
        # Save report / 保存報告
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        report_file = f"comparison_report_{timestamp}.html"
        
        report.show_html(report_file)
        print(f"✅ Comparison report saved as: {report_file}")
        
        # Display summary information / 顯示摘要信息
        print(f"\n📊 {report_title}")
        print(f"📈 Dataset 1 ({data1_name}): {data1.shape}")
        if data2 is not None:
            print(f"📈 Dataset 2 ({data2_name}): {data2.shape}")
        
        return report, report_file
        
    except ImportError:
        print("❌ sweetviz not available. Please install with: pip install sweetviz")
        return None, None
    except Exception as e:
        print(f"❌ Error generating comparison report: {e}")
        return None, None

# Generate comparison report / 生成比較報告
if stock_data is not None:
    # Create processed data for comparison / 創建處理後的數據進行比較
    processed_data = stock_data.copy()
    
    # Add some processing steps / 添加一些處理步驟
    processed_data['Returns'] = processed_data['Close'].pct_change()
    processed_data['Log_Returns'] = np.log(processed_data['Close'] / processed_data['Close'].shift(1))
    processed_data['MA5'] = processed_data['Close'].rolling(5).mean()
    processed_data['MA20'] = processed_data['Close'].rolling(20).mean()
    processed_data['Volatility'] = processed_data['Returns'].rolling(20).std()
    
    # Remove NaN values / 移除NaN值
    processed_data = processed_data.dropna()
    
    print(f"📊 Original data: {stock_data.shape}")
    print(f"📈 Processed data: {processed_data.shape}")
    
    # Generate comparison report / 生成比較報告
    comparison_report, comp_report_file = generate_comparison_report(
        stock_data, 
        processed_data,
        "Original 0700.HK Data",
        "Processed 0700.HK Data"
    )
else:
    print("❌ No data available for comparison report")

## 🛠️ 數據清理建議和自動化工具 / Data Cleaning Suggestions and Automation Tools

In [ ]:
class DataCleaningRecommender:
    """數據清理建議器 / Data Cleaning Recommender"""
    
    def __init__(self, data):
        self.data = data
        self.recommendations = []
    
    def analyze_and_recommend(self):
        """分析並提供清理建議 / Analyze and provide cleaning recommendations"""
        print("🛠️ Data Cleaning Recommendations / 數據清理建議")
        print("=" * 60)
        
        # Check for missing values / 檢查缺失值
        missing_values = self.data.isnull().sum()
        if missing_values.sum() > 0:
            self.recommendations.append({
                'issue': 'Missing values detected',
                'severity': 'Medium',
                'suggestion': 'Consider using forward/backward fill or interpolation for time series data',
                'code': 'df.fillna(method="ffill").fillna(method="bfill")'
            })
            
            for col, missing in missing_values[missing_values > 0].items():
                missing_pct = missing / len(self.data) * 100
                if missing_pct > 50:
                    self.recommendations.append({
                        'issue': f'High missing rate in {col} ({missing_pct:.1f}%)',
                        'severity': 'High',
                        'suggestion': f'Consider dropping {col} column',
                        'code': f'df.drop("{col}", axis=1)'
                    })
        
        # Check for duplicates / 檢查重複值
        duplicates = self.data.duplicated().sum()
        if duplicates > 0:
            self.recommendations.append({
                'issue': f'Duplicate rows found: {duplicates}',
                'severity': 'Medium',
                'suggestion': 'Remove duplicate rows',
                'code': 'df.drop_duplicates(inplace=True)'
            })
        
        # Check for outliers / 檢查異常值
        numeric_cols = self.data.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            values = self.data[col].dropna()
            if len(values) > 0:
                Q1 = values.quantile(0.25)
                Q3 = values.quantile(0.75)
                IQR = Q3 - Q1
                outliers = ((values < (Q1 - 1.5 * IQR)) | (values > (Q3 + 1.5 * IQR))).sum()
                outlier_pct = outliers / len(values) * 100
                
                if outlier_pct > 5:
                    self.recommendations.append({
                        'issue': f'High outlier percentage in {col}: {outlier_pct:.1f}%',
                        'severity': 'Low',
                        'suggestion': 'Consider outlier treatment (winsorization or removal)',
                        'code': f'# Outlier treatment for {col}\nQ1 = df["{col}"].quantile(0.25)\nQ3 = df["{col}"].quantile(0.75)\nIQR = Q3 - Q1\nlower_bound = Q1 - 1.5 * IQR\nupper_bound = Q3 + 1.5 * IQR\ndf["{col}"] = df["{col}"].clip(lower_bound, upper_bound)'
                    })
        
        # Check data types / 檢查數據類型
        for col in self.data.columns:
            if self.data[col].dtype == 'object':
                unique_ratio = self.data[col].nunique() / len(self.data)
                if unique_ratio > 0.5:
                    self.recommendations.append({
                        'issue': f'High cardinality in {col} ({self.data[col].nunique()} unique values)',
                        'severity': 'Low',
                        'suggestion': 'Consider encoding or grouping categorical variables',
                        'code': f'# Consider label encoding or one-hot encoding for {col}'
                    })
        
        # Check for time series specific issues / 檢查時間序列特定問題
        if isinstance(self.data.index, pd.DatetimeIndex):
            # Check for irregular frequency / 檢查不規則頻率
            inferred_freq = pd.infer_freq(self.data.index)
            if inferred_freq is None:
                self.recommendations.append({
                    'issue': 'Irregular time series frequency',
                    'severity': 'Medium',
                    'suggestion': 'Consider resampling to regular frequency',
                    'code': 'df_resampled = df.resample("D").mean()'
                })
        
        # Display recommendations / 顯示建議
        if not self.recommendations:
            print("✅ No major data cleaning issues detected!")
            print("🎉 Your data looks clean and ready for analysis!")
        else:
            for i, rec in enumerate(self.recommendations, 1):
                severity_emoji = {
                    'High': '🔴',
                    'Medium': '🟡', 
                    'Low': '🟢'
                }
                
                print(f"\n{i}. {severity_emoji.get(rec['severity'], '⚪')} {rec['issue']}")
                print(f"   Severity: {rec['severity']}")
                print(f"   Suggestion: {rec['suggestion']}")
                print(f"   Code example:\n   {rec['code']}")
        
        return self.recommendations
    
    def generate_cleaning_script(self):
        """生成清理腳本 / Generate cleaning script"""
        if not self.recommendations:
            return "# No cleaning needed - data is already clean!"
        
        script = [
            "#!/usr/bin/env python3",
            '"""",
            'Auto-generated data cleaning script',
            '自動生成的數據清理腳本',
            '"""',
            '',
            'import pandas as pd',
            'import numpy as np',
            '',
            'def clean_data(df):',
            '    """Clean the data based on recommendations"""',
            '    df_cleaned = df.copy()',
            ''
        ]
        
        # Add cleaning steps based on recommendations
        for rec in self.recommendations:
            if 'drop_duplicates' in rec['code']:
                script.append('    # Remove duplicates')
                script.append('    df_cleaned.drop_duplicates(inplace=True)')
                script.append('')
            elif 'drop(' in rec['code']:
                # Extract column name
                col_name = rec['code'].split('"')[1]
                script.append(f'    # Drop column with high missing rate: {col_name}')
                script.append(f'    if "{col_name}" in df_cleaned.columns:')
                script.append(f'        df_cleaned.drop("{col_name}", axis=1, inplace=True)')
                script.append('')
            elif 'fillna' in rec['code']:
                script.append('    # Fill missing values')
                script.append('    df_cleaned.fillna(method="ffill", inplace=True)')
                script.append('    df_cleaned.fillna(method="bfill", inplace=True)')
                script.append('')
        
        script.extend([
            '    print(f"Original shape: {df.shape}")',
            '    print(f"Cleaned shape: {df_cleaned.shape}")',
            '    ',
            '    return df_cleaned',
            '',
            '# Example usage:',
            '# df_clean = clean_data(your_dataframe)',
        ])
        
        return '\n'.join(script)

# Generate cleaning recommendations / 生成清理建議
if stock_data is not None:
    recommender = DataCleaningRecommender(stock_data)
    recommendations = recommender.analyze_and_recommend()
    
    # Generate cleaning script / 生成清理腳本
    cleaning_script = recommender.generate_cleaning_script()
    
    print("\n🔧 Auto-generated Cleaning Script:")
    print("=" * 50)
    print(cleaning_script)
else:
    print("❌ No data available for cleaning recommendations")

## 📋 整合報告總結 / Integrated Report Summary

In [ ]:
def generate_final_summary(data, quality_report, recommendations):
    """生成最終報告總結 / Generate Final Report Summary"""
    
    print("📊 Data Cleaning Tools Integration - Final Report")
    print("=" * 60)
    print("數據清理工具包集成 - 最終報告")
    print("=" * 60)
    
    # Data summary / 數據摘要
    print(f"\n📈 Data Summary / 數據摘要:")
    print(f"   Dataset: 0700.HK Trading Data")
    print(f"   Records: {len(data):,}")
    print(f"   Columns: {len(data.columns)}")
    print(f"   Date Range: {data.index[0].date()} to {data.index[-1].date()}")
    print(f"   Memory Usage: {data.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")
    
    # Quality metrics / 質量指標
    print(f"\n🔍 Quality Metrics / 質量指標:")
    if quality_report:
        missing_pct = quality_report.get('missing_percentage', 0)
        duplicates = quality_report.get('duplicates', 0)
        print(f"   Missing Values: {missing_pct:.2f}%")
        print(f"   Duplicate Rows: {duplicates}")
        print(f"   Data Completeness: {100 - missing_pct:.2f}%")
    
    # Tools status / 工具狀態
    print(f"\n🛠️ Tools Integration Status / 工具集成狀態:")
    
    tools_status = {
        'ydata-profiling': '✅ Available' if 'ydata_profiling' in sys.modules else '❌ Not Installed',
        'missingno': '✅ Available' if 'missingno' in sys.modules else '❌ Not Installed',
        'sweetviz': '✅ Available' if 'sweetviz' in sys.modules else '❌ Not Installed',
        'plotly': '✅ Available',
        'seaborn': '✅ Available',
        'matplotlib': '✅ Available'
    }
    
    for tool, status in tools_status.items():
        print(f"   {tool}: {status}")
    
    # Recommendations summary / 建議摘要
    print(f"\n💡 Recommendations Summary / 建議摘要:")
    if recommendations:
        high_priority = len([r for r in recommendations if r.get('severity') == 'High'])
        medium_priority = len([r for r in recommendations if r.get('severity') == 'Medium'])
        low_priority = len([r for r in recommendations if r.get('severity') == 'Low'])
        
        print(f"   High Priority Issues: {high_priority}")
        print(f"   Medium Priority Issues: {medium_priority}")
        print(f"   Low Priority Issues: {low_priority}")
    else:
        print("   ✅ No critical issues found")
    
    # Next steps / 下一步
    print(f"\n🚀 Next Steps / 下一步:")
    print("   1. Install missing tools: pip install -r enhanced_requirements.txt")
    print("   2. Review and apply cleaning recommendations")
    print("   3. Generate detailed reports with pandas-profiling")
    print("   4. Use sweetviz for comparative analysis")
    print("   5. Integrate with JupyterLab workflow")
    
    # Benefits achieved / 實現的收益
    print(f"\n🎯 Benefits Achieved / 實現的收益:")
    print("   ✅ Comprehensive data quality assessment")
    print("   ✅ Automated cleaning recommendations")
    print("   ✅ Professional visualization reports")
    print("   ✅ Integrated Simplified System data sources")
    print("   ✅ Bilingual support (Chinese/English)")
    print("   ✅ Ready for quantitative analysis")

# Generate final summary / 生成最終摘要
if stock_data is not None:
    generate_final_summary(
        stock_data, 
        basic_quality if 'basic_quality' in locals() else None,
        recommendations if 'recommendations' in locals() else []
    )
else:
    print("❌ No data available for final summary")

print("\n🎉 Data Cleaning Tools Integration Completed!")
print("🏁 數據清理工具包集成完成！")